# Phase I — Dataset Feasibility Notebook

This notebook documents how we collected/downloaded/cleaned/aggregated datasets for our semester project and exports one analysis-ready dataframe (`analysis_ready_phase1.csv`).

**Project GitHub repository:** https://github.com/VincentPit/ds_in_da_wild

**Collaborators:** Junyi Li(jl4724), Jinyue Wang(jw2796), Wenzhuo Zhang(wz475), Qiaohao Hu(qh252)

## 1) Research questions

**Primary question:**
- How does roadway traffic volume relate to motor vehicle collision frequency and injury severity in NYC?

**Supporting questions (Phase I — date-level join):**
1. On high-traffic days, are there systematically more crashes, injuries, and fatalities?
2. Does ambient air temperature (hyperlocal street-sensor data) modify the traffic–crash relationship?

**Expanded questions for Phase II — multi-dimensional analysis:**
3. Do crash rates and injury severity vary across **boroughs** after controlling for traffic volume? *(spatial heterogeneity)*
4. Are there strong **day-of-week** and **seasonal** patterns in traffic volume and crash outcomes that must be modeled as controls?
5. Does the composition of crash victims (pedestrians vs. cyclists vs. motorists) shift under different traffic-volume or temperature regimes?
6. Can we identify **high-risk combinations** (e.g., high traffic volume + extreme temperature + specific borough) that predict disproportionately severe outcomes?

## 2) Raw data description

Datasets used (NYC Open Data) — all CSVs stored in `data/`:

| # | Dataset | Source ID | Shape | Time Span | One row represents |
|---|---------|-----------|-------|-----------|-------------------|
| 1 | Traffic Volume Counts (Historical) | `btm5-ppia` | 42,756 rows × 31 cols | Jan 2012 – May 2021 | Directional hourly vehicle counts for one road segment on one date |
| 2 | Motor Vehicle Collisions – Crashes | `h9gi-nx95` | 2,244,212 rows × 29 cols | Jul 2012 – Feb 2026 | One reported collision citywide |
| 3 | NYC Climate Projections: Temperature & Precipitation | — | 108 rows × 8 cols | Baseline + decadal projections (2030s–2100) | One monthly/annual summary period |
| 4 | Hyperlocal Temperature Monitoring | — | 2,097,150 rows × 10 cols | Jun 2018 – Oct 2019 | One hourly air temperature reading from a single street-level sensor |

**Key column notes:**
- **Traffic:** 24 hourly count columns (`12:00-1:00 AM` … `11:00-12:00 AM`), plus `SegmentID`, `Roadway Name`, and `Direction`. Covers 1,956 unique road segments across 1,276 named roadways.
- **Crashes:** `COLLISION_ID` (unique key), `BOROUGH` (all 5 boroughs), `CRASH DATE/TIME`, and injured/killed counts broken down by pedestrian, cyclist, and motorist.
- **Climate Projections:** Monthly and annual baseline statistics (mean temperature, precipitation) plus projected changes under SSP2-4.5 and SSP5-8.5 scenarios through 2100. This dataset is reserved for future phases and is **not** joined at the daily level in Phase I.
- **Hyperlocal Temperature:** `Sensor.ID` (423 unique sensors mounted on street trees and light poles across 4 boroughs — Brooklyn, Bronx, Manhattan, Queens), `AirTemp` (°F), `Day`, `Hour`, `Borough`, and `ntacode` (neighborhood tabulation area). "Hyperlocal" refers to readings from low-cost sensors at street level, as opposed to airport-station weather records.

GitHub repository: https://github.com/VincentPit/ds_in_da_wild

In [10]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

data_dir = Path('data')
csv_files = sorted(data_dir.glob('*.csv'))

def normalized_name(p: Path) -> str:
    return re.sub(r'[^a-z0-9]+', '_', p.name.lower())

# Expected files (current project)
file_map = {}
for p in csv_files:
    n = normalized_name(p)
    if 'traffic_volume_counts' in n and 'historical' in n:
        file_map['traffic'] = p
    elif 'motor_vehicle_collisions' in n and 'crashes' in n:
        file_map['crashes'] = p
    elif 'climate_projections' in n and 'temperature_and_precipitation' in n:
        file_map['climate_projections'] = p
    elif 'hyperlocal_temperature_monitoring' in n:
        file_map['hyperlocal_temp'] = p

print('Detected CSV files:')
for p in csv_files:
    print('-', p.name)

required = ['traffic', 'crashes', 'climate_projections', 'hyperlocal_temp']
missing = [k for k in required if k not in file_map]
if missing:
    raise FileNotFoundError(f'Missing expected datasets: {missing}')

print('\nMapped files:')
for k, v in file_map.items():
    print(f'{k:20s} ---->> {v.name}')

Detected CSV files:
- Hyperlocal_Temperature_Monitoring_20260225.csv
- Motor_Vehicle_Collisions_-_Crashes_20260225.csv
- New_York_City_Climate_Projections__Temperature_and_Precipitation_20260225.csv
- Traffic_Volume_Counts_(Historical)_20260225.csv

Mapped files:
hyperlocal_temp      ---->> Hyperlocal_Temperature_Monitoring_20260225.csv
crashes              ---->> Motor_Vehicle_Collisions_-_Crashes_20260225.csv
climate_projections  ---->> New_York_City_Climate_Projections__Temperature_and_Precipitation_20260225.csv
traffic              ---->> Traffic_Volume_Counts_(Historical)_20260225.csv


In [11]:
# Load raw datasets
traffic = pd.read_csv(file_map['traffic'])
crashes = pd.read_csv(file_map['crashes'], low_memory=False)
climate_proj = pd.read_csv(file_map['climate_projections'])
hyperlocal = pd.read_csv(file_map['hyperlocal_temp'])

print('Shapes:')
print('traffic         :', traffic.shape)
print('crashes         :', crashes.shape)
print('climate_proj    :', climate_proj.shape)
print('hyperlocal_temp :', hyperlocal.shape)

# Standardize key date columns
traffic['Date'] = pd.to_datetime(traffic.get('Date'), errors='coerce')
crashes['CRASH DATE'] = pd.to_datetime(crashes.get('CRASH DATE'), errors='coerce')

# Crash numeric cleaning
num_cols = [
    'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED'
 ]
for c in num_cols:
    if c in crashes.columns:
        crashes[c] = pd.to_numeric(crashes[c], errors='coerce').fillna(0)

# Traffic hourly columns -> daily volume
hourly_cols = [
    c for c in traffic.columns
    if re.match(r'^\d{1,2}:\d{2}-\d{1,2}:\d{2}\s?[AP]M$', str(c).strip())
 ]
if not hourly_cols:
    raise ValueError('No hourly traffic columns detected in traffic dataset.')

for c in hourly_cols:
    traffic[c] = pd.to_numeric(traffic[c], errors='coerce')
traffic['daily_volume'] = traffic[hourly_cols].sum(axis=1, min_count=1).fillna(0)

display(traffic.head(2))
display(crashes.head(2))

Shapes:
traffic         : (42756, 31)
crashes         : (2244212, 29)
climate_proj    : (108, 8)
hyperlocal_temp : (2097150, 10)


,ID,SegmentID,Roadway Name,From,To,Direction,Date,12:00-1:00 AM,1:00-2:00AM,2:00-3:00AM,3:00-4:00AM,4:00-5:00AM,5:00-6:00AM,6:00-7:00AM,7:00-8:00AM,8:00-9:00AM,9:00-10:00AM,10:00-11:00AM,11:00-12:00PM,12:00-1:00PM,1:00-2:00PM,2:00-3:00PM,3:00-4:00PM,4:00-5:00PM,5:00-6:00PM,6:00-7:00PM,7:00-8:00PM,8:00-9:00PM,9:00-10:00PM,10:00-11:00PM,11:00-12:00AM,daily_volume
0,1,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-09,20.0,10.0,11.0,14.0,13.0,20.0,34.0,66.0,100.0,52.0,68.0,85.0,85.0,94.0,104.0,105.0,147.0,120.0,91.0,83.0,74.0,49.0,42.0,42.0,1529.0
1,2,15540,BEACH STREET,UNION PLACE,VAN DUZER STREET,NB,2012-01-10,21.0,16.0,8.0,6.0,13.0,13.0,31.0,70.0,67.0,45.0,57.0,67.0,73.0,95.0,102.0,98.0,133.0,131.0,95.0,73.0,70.0,63.0,42.0,35.0,1424.0


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,2021-09-11,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,2.0,0.0,0,0,0,0,2,0,Aggressive Driving/Road Rage,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,2022-03-26,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,1.0,0.0,0,0,0,0,1,0,Pavement Slippery,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN


In [12]:
# Aggregate and merge into one analysis-ready dataframe (daily level)

# 1) Daily traffic totals
traffic_daily = (
    traffic.groupby('Date', as_index=False)
    .agg(
        total_daily_volume=('daily_volume', 'sum'),
        traffic_segment_rows=('daily_volume', 'size')
    )
)

# 2) Daily crash outcomes
crashes_daily = (
    crashes.groupby('CRASH DATE', as_index=False)
    .agg(
        daily_crashes=('COLLISION_ID', 'nunique'),
        persons_injured=('NUMBER OF PERSONS INJURED', 'sum'),
        persons_killed=('NUMBER OF PERSONS KILLED', 'sum')
    )
    .rename(columns={'CRASH DATE': 'Date'})
)

# 3) Hyperlocal daily average temperature (auto-detect date + temp-like columns)
hyper = hyperlocal.copy()
date_candidates = [c for c in hyper.columns if 'date' in c.lower() or 'time' in c.lower()]
temp_candidates = [c for c in hyper.columns if 'temp' in c.lower()]

if date_candidates:
    hyper['Date'] = pd.to_datetime(hyper[date_candidates[0]], errors='coerce')
else:
    hyper['Date'] = pd.NaT

if temp_candidates:
    hyper[temp_candidates[0]] = pd.to_numeric(hyper[temp_candidates[0]], errors='coerce')
    hyper_daily = hyper.groupby('Date', as_index=False)[temp_candidates[0]].mean()
    hyper_daily = hyper_daily.rename(columns={temp_candidates[0]: 'hyperlocal_temp_daily_avg'})
else:
    hyper_daily = pd.DataFrame({'Date': pd.to_datetime([]), 'hyperlocal_temp_daily_avg': []})

# 4) Merge all
analysis_ready = traffic_daily.merge(crashes_daily, on='Date', how='inner')
analysis_ready = analysis_ready.merge(hyper_daily, on='Date', how='left')
analysis_ready = analysis_ready.sort_values('Date').reset_index(drop=True)

# Optional engineered columns
analysis_ready['injuries_per_crash'] = np.where(
    analysis_ready['daily_crashes'] > 0,
    analysis_ready['persons_injured'] / analysis_ready['daily_crashes'],
    0.0
)
analysis_ready['fatalities_per_crash'] = np.where(
    analysis_ready['daily_crashes'] > 0,
    analysis_ready['persons_killed'] / analysis_ready['daily_crashes'],
    0.0
)

print('analysis_ready shape:', analysis_ready.shape)
display(analysis_ready.head())

analysis_ready shape: (561, 9)


,Date,total_daily_volume,traffic_segment_rows,daily_crashes,persons_injured,persons_killed,hyperlocal_temp_daily_avg,injuries_per_crash,fatalities_per_crash
0,2012-09-28,72452.0,11,703,188.0,2.0,NaN,0.267425,0.002845
1,2012-09-29,340468.0,52,485,134.0,1.0,NaN,0.276289,0.002062
2,2012-09-30,1153866.5,173,449,145.0,2.0,NaN,0.322940,0.004454
3,2012-10-01,1162691.0,173,555,162.0,1.0,NaN,0.291892,0.001802
4,2012-10-02,1160570.0,173,593,191.0,0.0,NaN,0.322091,0.000000


In [13]:
# Export final analysis-ready dataframe
output_path = Path('analysis_ready_phase1.csv')
analysis_ready.to_csv(output_path, index=False)

print('Exported:', output_path.resolve())
print('Columns:', list(analysis_ready.columns))
print('Row count:', len(analysis_ready))

Exported: /Users/stephenlee/Desktop/WildDs/ds_in_da_wild/analysis_ready_phase1.csv
Columns: ['Date', 'total_daily_volume', 'traffic_segment_rows', 'daily_crashes', 'persons_injured', 'persons_killed', 'hyperlocal_temp_daily_avg', 'injuries_per_crash', 'fatalities_per_crash']
Row count: 561


## 3) Plain-English summary of code steps

**Cell 1 — File detection** (`data/` folder scan):
Scans the `data/` directory and maps each CSV to a logical key (`traffic`, `crashes`, `climate_projections`, `hyperlocal_temp`) using filename pattern matching. Raises an error immediately if any expected file is missing, so downstream cells can safely assume all four datasets are available.

**Cell 2 — Loading & type coercion:**
Reads all four CSVs into DataFrames. Key operations:
- Parses `Date` (traffic) and `CRASH DATE` (crashes) into proper datetime objects.
- Converts all eight injury/fatality count columns in crashes to numeric, filling missing values with 0.
- Detects the 24 hourly traffic-count columns by regex (e.g., `12:00-1:00 AM`, `5:00-6:00PM`) and sums them into a new `daily_volume` column — one road-segment/direction row now carries its full-day vehicle count.

**Cell 3 — Daily aggregation & merge:**
1. *Traffic → daily totals:* groups by `Date`, sums `daily_volume` into `total_daily_volume`, and counts the number of segment rows contributing (`traffic_segment_rows`).
2. *Crashes → daily outcomes:* groups by `CRASH DATE`, counts unique `COLLISION_ID`s, and totals `persons_injured` and `persons_killed`.
3. *Hyperlocal temperature → daily mean:* auto-detects the date column (`Day`) and temperature column (`AirTemp`); groups by date and averages across all sensors active that day.
4. *Merge:* traffic and crashes are joined on `Date` with an **inner join** (only dates present in both datasets are kept). The temperature data is left-joined, so rows outside the sensor deployment window (Jun 2018 – Oct 2019) receive `NaN` for `hyperlocal_temp_daily_avg`.

**Cell 4 — Feature engineering & export:**
Adds two rate features — `injuries_per_crash` and `fatalities_per_crash` — computed as injured/killed totals divided by daily crash count (set to 0 on days with no crashes). Exports the final dataframe to `analysis_ready_phase1.csv`.

## 4) Analysis-ready dataframe description

- **Unit of observation:** one row per calendar date where both traffic-volume records and crash records exist (inner join on `Date`).
- **Effective temporal coverage:** the overlapping window of the traffic data (Jan 2012 – May 2021) and crash data (Jul 2012 – Feb 2026) → **Jul 2012 – May 2021**.

| Column | Source | Description |
|--------|--------|-------------|
| `Date` | Shared join key | Calendar date |
| `total_daily_volume` | Traffic | Sum of all hourly vehicle counts across every monitored road segment for that date |
| `traffic_segment_rows` | Traffic | Number of segment-direction records contributing to that day's total (proxy for data coverage) |
| `daily_crashes` | Crashes | Count of unique collision IDs reported citywide that day |
| `persons_injured` | Crashes | Total persons injured (pedestrians + cyclists + motorists combined) |
| `persons_killed` | Crashes | Total persons killed |
| `hyperlocal_temp_daily_avg` | Hyperlocal sensors | Mean air temperature (°F) averaged across all active **street-level sensors** on that date. "Hyperlocal" refers to readings from 423 low-cost sensors mounted on street trees and light poles in 4 boroughs (Brooklyn, Bronx, Manhattan, Queens) — not airport-station data. Only populated for **Jun 2018 – Oct 2019**; `NaN` outside that window (~85% of rows). |
| `injuries_per_crash` | Engineered | `persons_injured / daily_crashes` (0.0 when no crashes that day) |
| `fatalities_per_crash` | Engineered | `persons_killed / daily_crashes` (0.0 when no crashes that day) |

This dataframe is the primary input for Phase II EDA and modeling. Temperature-related analyses should be scoped explicitly to the Jun 2018 – Oct 2019 sub-window.

## 5) Limitations & project scope considerations

**Partial network coverage (traffic data):**
The traffic dataset covers ~1,956 monitored road segments out of NYC's full street network. Daily volume totals reflect changes in *monitored* traffic, not citywide throughput. Segments are not uniformly distributed across boroughs, which can bias borough-level comparisons.

**Narrow hyperlocal sensor window:**
The street-sensor network was only active for ~15 months (Jun 2018 – Oct 2019) and covers 4 of 5 boroughs (Staten Island absent). As a result, `hyperlocal_temp_daily_avg` is `NaN` for roughly 85% of the merged dataset. Any temperature-based analysis must be scoped to this sub-window and treated as a **separate, clearly bounded sub-analysis** rather than a core finding.

**Date-level spatial aggregation:**
Phase I collapses all records to the daily city-wide level, masking borough-level, street-level, and hour-of-day variation. Phase II should disaggregate by borough (a borough × date panel) and consider adding hour-of-day features to capture peak-hour effects.

**Calendar confounding:**
Both traffic volume and crash counts have well-documented **day-of-week** and **seasonal** patterns (e.g., Friday spikes, summer vs. winter baseline). Failing to control for these can make temperature or volume appear more/less significant than it truly is. Phase II models must include day-of-week dummies, month or quarter fixed effects, and a public-holiday indicator at a minimum.

**Count-data modeling:**
`daily_crashes` is a non-negative integer count; OLS regression is inappropriate. The correct baseline is **Poisson regression**, followed by a dispersion test. Crash count variance typically exceeds the mean (overdispersion), in which case **Negative Binomial** becomes the preferred main model. Both should be reported for comparison.

**Causal interpretation:**
Observed correlations between volume and crashes partly reflect *exposure* (more vehicles → higher baseline collision probability) rather than a causal effect of traffic intensity. Phase II should consider crash-rate metrics (crashes per unit volume) or, ideally, explore quasi-experimental designs to sharpen causal claims.

## 6) Questions for TA reviewer

1. **Model selection sequence:** We plan to start with a **Poisson regression** baseline for `daily_crashes`, then run a dispersion test. If overdispersion is confirmed (variance > mean, which we expect for crash counts), we'll use **Negative Binomial** as the primary model. Should we report both side-by-side, or only the better-fitting model?

2. **Calendar controls:** Given the strong day-of-week and seasonal structure in both traffic and crashes, we intend to include day-of-week dummies, month or quarter fixed effects, and a public-holiday indicator. Are there other temporal controls you'd recommend (e.g., year fixed effects for long-run trend, COVID-period indicator for 2020)?

3. **Spatial disaggregation:** Should Phase II use a citywide daily-level model, or would a **borough × date panel** with borough fixed effects be more defensible given the heterogeneity across boroughs in both traffic coverage and crash rates?

4. **Temperature sub-analysis scope:** Since `hyperlocal_temp_daily_avg` is only available for Jun 2018 – Oct 2019 (~15 months), should we present the temperature analysis as an explicitly scoped sub-analysis with its own sample description, rather than folding it into the main model?